# BYOL — Bootstrap Your Own Latent: A New Approach to Self-Supervised Learning (2020)

_Papers · Self-supervised & Representation_

**Learn image features with no labels AND no negative pairs — one network predicts a slow-moving copy of itself.**

---

This notebook builds BYOL from scratch, one step at a time. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns**. We pull a few raw samples from the MNIST dataset to see them before any transforms.

In [ ]:
import torchvision

preview = torchvision.datasets.MNIST(root="./data", train=True, download=True)
print("dataset: MNIST   samples:", len(preview))
first_image, first_label = preview[0]
print("one sample:", first_image.size, "image,  label =", first_label)
print("classes:", getattr(preview, "classes", "(digit labels 0-9)"))
samples = [preview[i] for i in range(5)]
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

We build BYOL piece by piece: (1) check the prediction loss and EMA step on a worked example, (2) define the encoder, projector, and predictor, (3) assemble the online/target pair with stop-gradient + EMA and pretrain on unlabelled MNIST, (4) freeze the encoder and probe its features, and (5) ablate the predictor to watch the representation collapse.

### Step 1 — Sanity-check the loss and the EMA update

BYOL's loss is the squared error between L2-normalized prediction and target vectors, which equals `2 - 2*cos(p, z)` (Eqn. 2). The target network is a slow exponential moving average (EMA) of the online network: `xi <- tau*xi + (1-tau)*theta` (Eqn. 1). We verify both on small numbers before wiring up the full model.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import copy
import torchvision
import torchvision.transforms as T

torch.manual_seed(0)
np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Prediction loss (Eqn. 2): squared error of normalized vectors == 2 - 2*cos.
a = torch.tensor([3.0, 4.0])     # online prediction (pre-normalize)
b = torch.tensor([4.0, 3.0])     # target (pre-normalize)
loss = ((F.normalize(a, dim=0) - F.normalize(b, dim=0))**2).sum()
cos = (a @ b) / (a.norm() * b.norm())
print("worked example:  cos =", round(cos.item(), 4),
      " loss =", round(loss.item(), 4), " (== 2-2cos =", round((2 - 2*cos).item(), 4), ")")

# One EMA step (Eqn. 1) for the target parameters.
tau_ema = 0.99
xi, th = 1.00, 2.00
new_xi = tau_ema*xi + (1 - tau_ema)*th
print("EMA step:  xi <- tau*xi + (1-tau)*theta =", round(new_xi, 4))
# worked example:  cos = 0.96  loss = 0.08  (== 2-2cos = 0.08 )
# EMA step:  xi <- tau*xi + (1-tau)*theta = 1.01

### Step 2 — Define the building blocks (encoder, projector, predictor)

Three small networks make up BYOL. The **encoder** `f` is a tiny conv net that maps an image to a representation. The **projector** `g` and **predictor** `q` are one-hidden-layer MLPs. The predictor lives on the online side only — that asymmetry is what stops the network from collapsing to a constant.

In [ ]:
class Encoder(nn.Module):                  # small conv net -> representation
    def __init__(self, feat=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.fc = nn.Linear(32, feat)

    def forward(self, x):
        return F.relu(self.fc(self.net(x)))


def mlp(fin=64, hid=128, out=64):          # projector / predictor: MLP, one hidden layer + ReLU
    return nn.Sequential(nn.Linear(fin, hid), nn.ReLU(), nn.Linear(hid, out))

### Step 3 — Assemble the online/target pair and pretrain on unlabelled MNIST

The `BYOL` module holds an **online** branch (encoder + projector + predictor) and a **target** branch (encoder + projector) that is a no-gradient EMA copy. Each step we feed two random augmentations of the same images, push the online prediction toward the stop-gradient target (symmetrized over the two views), take the optimizer step, then update the target by EMA. No negative pairs are used anywhere.

In [ ]:
class BYOL(nn.Module):
    def __init__(self, use_predictor=True, tau=0.99):
        super().__init__()
        self.tau = tau
        self.use_predictor = use_predictor
        self.enc_o, self.proj_o = Encoder(), mlp()          # online encoder + projector
        self.pred = mlp() if use_predictor else nn.Identity()  # predictor: ONLINE side only
        self.enc_t = copy.deepcopy(self.enc_o)              # target = EMA copy (no grad)
        self.proj_t = copy.deepcopy(self.proj_o)
        for p in list(self.enc_t.parameters()) + list(self.proj_t.parameters()):
            p.requires_grad_(False)

    def online(self, x):
        return self.pred(self.proj_o(self.enc_o(x)))        # q(g(f(x)))

    def target(self, x):
        with torch.no_grad():                               # stop-gradient
            return self.proj_t(self.enc_t(x))

    def loss(self, v1, v2):                 # normalized-MSE (Eqn. 2), symmetrized
        def nmse(p, z):
            return ((F.normalize(p, dim=1) - F.normalize(z, dim=1))**2).sum(1).mean()
        return nmse(self.online(v1), self.target(v2)) + nmse(self.online(v2), self.target(v1))

    @torch.no_grad()
    def ema_update(self):                   # Eqn. 1: xi <- tau*xi + (1-tau)*theta
        for o, t in [(self.enc_o, self.enc_t), (self.proj_o, self.proj_t)]:
            for po, pt in zip(o.parameters(), t.parameters()):
                pt.mul_(self.tau).add_(po.detach(), alpha=1 - self.tau)


# Two-view augmentation + an UNLABELLED MNIST subset (torchvision, preinstalled).
aug = T.Compose([T.RandomResizedCrop(28, scale=(0.5, 1.0)),
                 T.RandomApply([T.GaussianBlur(3)], 0.5), T.ToTensor()])
base = T.ToTensor()
raw = torchvision.datasets.MNIST("./data", train=True, download=True)
idx = np.random.RandomState(0).permutation(len(raw))[:3000]
imgs = [raw[i][0] for i in idx]
labels = torch.tensor([raw[i][1] for i in idx])     # used ONLY for the probe later


def pretrain(use_predictor=True, epochs=15):
    torch.manual_seed(0)
    m = BYOL(use_predictor=use_predictor).to(device)
    opt = torch.optim.Adam([p for p in m.parameters() if p.requires_grad], lr=1e-3)
    m.train()
    B = 128
    for ep in range(epochs):
        perm = np.random.permutation(len(imgs))
        tot, nb_batches = 0.0, 0
        for s in range(0, len(imgs), B):
            bi = perm[s:s + B]
            v1 = torch.stack([aug(imgs[i]) for i in bi]).to(device)
            v2 = torch.stack([aug(imgs[i]) for i in bi]).to(device)
            loss = m.loss(v1, v2)
            opt.zero_grad()
            loss.backward()
            opt.step()
            m.ema_update()                              # online step THEN EMA
            tot += loss.item()
            nb_batches += 1
        if ep % 3 == 0:
            print(f"  pretrain ep {ep}  BYOL loss {tot/nb_batches:.4f}")
    return m


print("\n=== full BYOL (with predictor) ===")
m = pretrain(use_predictor=True)

### Step 4 — Freeze the encoder and probe its features (linear evaluation)

The standard test for self-supervised features: freeze the encoder and train only a linear classifier on a few labels. We compare that frozen-feature probe against a fresh conv net trained end-to-end on the same few labels. If BYOL learned useful structure, the probe should win — especially in the low-label regime.

In [ ]:
def features(model):
    model.eval()
    with torch.no_grad():
        all_imgs = torch.stack([base(im) for im in imgs]).to(device)
        return model.enc_o(all_imgs).cpu()


feats = features(m)
print("feature std across images (full BYOL):", round(feats.std(0).mean().item(), 4), "(healthy: > 0)")


def linear_probe(feats, n_lab):             # train ONLY a linear classifier on frozen features
    accs = []
    for seed in range(3):
        g = np.random.RandomState(seed)
        tr = g.permutation(len(labels))[:n_lab]
        te = g.permutation(len(labels))[-600:]
        clf = nn.Linear(feats.shape[1], 10)
        o = torch.optim.Adam(clf.parameters(), lr=0.05)
        for _ in range(200):
            o.zero_grad()
            F.cross_entropy(clf(feats[tr]), labels[tr]).backward()
            o.step()
        with torch.no_grad():
            accs.append((clf(feats[te]).argmax(1) == labels[te]).float().mean().item())
    return float(np.mean(accs))


def from_scratch(n_lab):                     # train a fresh conv net end-to-end on the few labels
    accs = []
    for seed in range(3):
        torch.manual_seed(seed)
        g = np.random.RandomState(seed)
        tr = g.permutation(len(labels))[:n_lab]
        te = g.permutation(len(labels))[-600:]
        net = nn.Sequential(Encoder(), nn.Linear(64, 10))
        o = torch.optim.Adam(net.parameters(), lr=1e-3)
        Xtr = torch.stack([base(imgs[i]) for i in tr])
        net.train()
        for _ in range(60):
            o.zero_grad()
            F.cross_entropy(net(Xtr), labels[tr]).backward()
            o.step()
        net.eval()
        with torch.no_grad():
            Xte = torch.stack([base(imgs[i]) for i in te])
            accs.append((net(Xte).argmax(1) == labels[te]).float().mean().item())
    return float(np.mean(accs))


print("\nlabels | probe(frozen BYOL) | from-scratch")
for n in [20, 50, 100, 300]:
    print(f"{n:6d} |       {linear_probe(feats, n):.3f}        |    {from_scratch(n):.3f}")

### Step 5 — Ablate the predictor and watch the representation collapse

Now remove the predictor so the online and target branches are symmetric. With no predictor (and no negatives forcing images apart), the encoder can drive the loss to zero by outputting the *same* constant vector for every image — a collapse. We re-pretrain, then check that the feature variance crashes toward 0 and the probe falls to near chance (~0.10 on 10-way MNIST).

In [ ]:
print("\n=== ablation: NO predictor (expect collapse) ===")
m_ab = pretrain(use_predictor=False)
feats_ab = features(m_ab)
print("feature std across images (no predictor):", round(feats_ab.std(0).mean().item(), 4), "(collapsed: ~0)")
print("probe(no-predictor BYOL) @100 labels:", round(linear_probe(feats_ab, 100), 3), "(~chance 0.10)")
# Full BYOL: probe beats from-scratch at every budget, features have healthy variance.
# No predictor: feature std collapses toward 0 and the probe falls to ~chance.
# (Exact numbers vary by hardware/seed; this is our small run, not the paper's reported result.)

## Visualize the data & results

_Does BYOL learn useful features with NO negatives — and does removing the predictor collapse them?_

### Step 1 — Redefine the model and load the unlabelled data

The visualization re-runs the whole experiment standalone so this cell block can stand on its own. We restate the encoder, projector/predictor MLP, and the BYOL module (online + target with EMA and stop-gradient), then load the same 3000-image unlabelled MNIST subset and the two-view augmentation.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import copy
import torchvision
import torchvision.transforms as T

torch.manual_seed(0)
np.random.seed(0)

# BYOL with NO negatives on UNLABELLED MNIST: freeze, probe vs from-scratch, and the
# no-predictor collapse ablation (toy reproduction).
class Encoder(nn.Module):
    def __init__(s, feat=64):
        super().__init__()
        s.net = nn.Sequential(nn.Conv2d(1,16,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
                              nn.Conv2d(16,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
                              nn.AdaptiveAvgPool2d(1), nn.Flatten())
        s.fc = nn.Linear(32, feat)
    def forward(s, x): return F.relu(s.fc(s.net(x)))
def mlp(fin=64, hid=128, out=64):
    return nn.Sequential(nn.Linear(fin,hid), nn.ReLU(), nn.Linear(hid,out))

class BYOL(nn.Module):
    def __init__(s, use_predictor=True, tau=0.99):
        super().__init__(); s.tau=tau
        s.enc_o, s.proj_o = Encoder(), mlp()
        s.pred = mlp() if use_predictor else nn.Identity()
        s.enc_t, s.proj_t = copy.deepcopy(s.enc_o), copy.deepcopy(s.proj_o)
        for p in list(s.enc_t.parameters())+list(s.proj_t.parameters()): p.requires_grad_(False)
    def online(s, x): return s.pred(s.proj_o(s.enc_o(x)))
    def target(s, x):
        with torch.no_grad(): return s.proj_t(s.enc_t(x))
    def loss(s, v1, v2):
        def nmse(p,z): return ((F.normalize(p,dim=1)-F.normalize(z,dim=1))**2).sum(1).mean()
        return nmse(s.online(v1), s.target(v2)) + nmse(s.online(v2), s.target(v1))
    @torch.no_grad()
    def ema(s):
        for o,t in [(s.enc_o,s.enc_t),(s.proj_o,s.proj_t)]:
            for po,pt in zip(o.parameters(), t.parameters()):
                pt.mul_(s.tau).add_(po.detach(), alpha=1-s.tau)

aug  = T.Compose([T.RandomResizedCrop(28, scale=(0.5,1.0)),
                  T.RandomApply([T.GaussianBlur(3)], 0.5), T.ToTensor()])
base = T.ToTensor()
raw  = torchvision.datasets.MNIST("./data", train=True, download=True)
idx  = np.random.RandomState(0).permutation(len(raw))[:3000]
imgs = [raw[i][0] for i in idx]; labels = torch.tensor([raw[i][1] for i in idx])

### Step 2 — Define the pretrain, probe, and from-scratch helpers

Same three procedures as the reference run: `pretrain` runs the online step then the EMA update each batch; `features` returns frozen-encoder outputs; `probe` trains only a linear head on those frozen features; and `scratch` trains a fresh conv net end-to-end. These are the pieces we compare in the next step.

In [ ]:
def pretrain(use_predictor=True, epochs=15):
    torch.manual_seed(0); m = BYOL(use_predictor=use_predictor)
    opt = torch.optim.Adam([p for p in m.parameters() if p.requires_grad], lr=1e-3)
    m.train(); B=128
    for ep in range(epochs):
        perm = np.random.permutation(len(imgs))
        for s0 in range(0, len(imgs), B):
            bi = perm[s0:s0+B]
            v1 = torch.stack([aug(imgs[i]) for i in bi]); v2 = torch.stack([aug(imgs[i]) for i in bi])
            loss = m.loss(v1, v2)
            opt.zero_grad(); loss.backward(); opt.step(); m.ema()
    return m
def features(m):
    m.eval()
    with torch.no_grad(): return m.enc_o(torch.stack([base(im) for im in imgs]))

def probe(feats, n):
    a=[]
    for seed in range(3):
        g=np.random.RandomState(seed); tr=g.permutation(len(labels))[:n]; te=g.permutation(len(labels))[-600:]
        clf=nn.Linear(feats.shape[1],10); o=torch.optim.Adam(clf.parameters(),lr=0.05)
        for _ in range(200): o.zero_grad(); F.cross_entropy(clf(feats[tr]),labels[tr]).backward(); o.step()
        with torch.no_grad(): a.append((clf(feats[te]).argmax(1)==labels[te]).float().mean().item())
    return round(float(np.mean(a)),3)
def scratch(n):
    a=[]
    for seed in range(3):
        torch.manual_seed(seed); g=np.random.RandomState(seed)
        tr=g.permutation(len(labels))[:n]; te=g.permutation(len(labels))[-600:]
        net=nn.Sequential(Encoder(), nn.Linear(64,10)); o=torch.optim.Adam(net.parameters(),lr=1e-3)
        Xtr=torch.stack([base(imgs[i]) for i in tr]); net.train()
        for _ in range(60): o.zero_grad(); F.cross_entropy(net(Xtr),labels[tr]).backward(); o.step()
        net.eval()
        with torch.no_grad():
            Xte=torch.stack([base(imgs[i]) for i in te]); a.append((net(Xte).argmax(1)==labels[te]).float().mean().item())
    return round(float(np.mean(a)),3)

### Step 3 — Run full BYOL vs from-scratch, then the no-predictor collapse

Finally we pretrain full BYOL, print probe-vs-from-scratch accuracy across label budgets and the healthy feature spread, then re-pretrain without the predictor and confirm the feature std collapses toward 0 and the probe drops to near chance — the headline result and its ablation side by side.

In [ ]:
m_full = pretrain(use_predictor=True); f_full = features(m_full)
for n in [20,50,100,300]:
    print(n, "probe", probe(f_full, n), "scratch", scratch(n))
print("full BYOL feature std:", round(f_full.std(0).mean().item(),3))

m_ab = pretrain(use_predictor=False); f_ab = features(m_ab)
print("no-predictor feature std:", round(f_ab.std(0).mean().item(),3), "(collapsed ~0)")
print("no-predictor probe @100:", probe(f_ab, 100), "(~chance)")
# Full BYOL probe > from-scratch at every budget (no negatives used).
# No predictor -> feature std collapses toward 0, probe near chance.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The headline (no negatives). You pretrained an encoder with BYOL on unlabelled images —
            no negative pairs at all — froze it, and trained a linear probe on just 20 labels; you
            also trained a from-scratch model on the same 20 labels. The probe scores much higher. What does
            that demonstrate, and how is it different from what SimCLR (paper-simclr) / MoCo (paper-moco)
            would have needed?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compare the two accuracies at the smallest label budget; the frozen-BYOL probe wins. — _With only 20 labels a from-scratch net has too little signal; the probe inherits features learned from thousands of unlabelled images._
- Note that the BYOL loss contained no negatives — only a positive-pair prediction against a slow EMA target. — _SimCLR/MoCo need many negatives (big batch / memory queue) to avoid collapse; BYOL replaces that with the predictor + stop-grad + EMA._
- Conclude that negative-free, label-free pretraining alone supplied the useful structure. — _Same encoder, same probe, same labels; only the pretraining differs — and it used no negatives._

**Answer:** It demonstrates that negative-free, label-free pretraining transfers: in the
                 low-label regime a linear probe on frozen BYOL features beats from-scratch because the
                 features were shaped by thousands of unlabelled images — using no negative pairs.
                 SimCLR / MoCo would have needed a large batch or a memory queue of negatives to avoid
                 collapse; BYOL instead relies on the predictor, the stop-gradient, and the slow EMA target.
                 Our CODEVIZ panel shows the probe beating from-scratch across the label budgets.

</details>

**Problem 2.** Your worked example gave $\mathcal{L} = 0.08$ for prediction $a=[3,4]$ and target $b=[4,3]$
            (cosine $0.96$). Now suppose training drifts so the prediction becomes $a=[5,0]$ while the target
            stays $b=[4,3]$. Does the loss go up or down, and by how much?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Lengths: $\lVert a\rVert_2=\sqrt{25}=5$, $\lVert b\rVert_2=\sqrt{16+9}=5$. — _Need both norms to normalize for the cosine form of Eqn. 2._
- Dot product $\langle a,b\rangle = 5\cdot4 + 0\cdot3 = 20$; cosine $=20/(5\cdot5)=0.80$. — _Cosine fell from $0.96$ to $0.80$ — prediction and target now point less alike._
- Loss $=2-2(0.80)=0.40$. — _Lower cosine → larger $2-2\cos$ → larger loss._

**Answer:** The loss goes up, from $0.08$ to $0.40$. The prediction and target are now less
                 aligned (cosine $0.96 \to 0.80$), and since the loss is $2-2\cos$, a smaller cosine means a
                 bigger loss. Gradient descent on $\theta$ would push the online prediction back toward the
                 target to shrink it again.

</details>

**Problem 3.** Ablation. You remove the predictor $q_\theta$ (so the online side is just encoder +
            projector, identical in shape to the target) and re-pretrain BYOL. Training loss drops to almost
            $0$ within a few steps, but the linear probe is now near chance (~10% on 10-way MNIST). What
            happened, and which design choice prevents it in real BYOL?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Inspect the encoder outputs across different images after the collapsed run. — _They are nearly identical — the encoder outputs almost the same vector for every image (a constant collapse)._
- See why the loss went to ~0: if both sides output the same constant, prediction = target exactly, so $2-2\cos = 0$. — _With no negatives and a symmetric online/target, the trivial constant solution minimizes the loss — nothing forces images apart._
- Restore the predictor $q_\theta$ (asymmetry) + stop-gradient + EMA target. — _The predictor and the lagging, stop-gradient target make the two sides hard to collapse jointly (paper §3, §5 Table 5b)._

**Answer:** The representation collapsed to a constant: with the predictor gone, the online and
                 target branches are symmetric, and outputting the same constant vector for every image makes
                 the normalized-MSE loss exactly $0$ while learning nothing — so the probe is at chance. Real
                 BYOL prevents this with the asymmetric predictor $q_\theta$ on the online side, the
                 stop-gradient on the target, and the slow EMA target — exactly the components
                 the paper's §5 ablation removes to reproduce the collapse. Our CODEVIZ panel shows the
                 collapsed (no-predictor) feature variance crushed toward $0$ versus the healthy full run.

</details>